#PySpark Practice Notebook

Experimenting further with Spark Dataframes with Museum Visits Data


In [ ]:
#Create CSV files for the following Datasets
"""
visitors
visitorId,firstName,lastName,age
101,Alice,Smith,30
102,Bob,Johnson,45
103,Charlie,Williams,50
104,John,Smith,78

visits
visitId,visitorId,visitDuration
1,101,15
2,101,30
3,102,45
4,102,30
5,104,20
6,103,60
7,103,50
8,104,45
9,,45
10,101,30
11,103,40
12,,30
13,104,25
15,102,15
"""

In [3]:
#Import necessary spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F 

In [2]:
#Create a SparkSession
spark=SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 07:02:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
#Create dataframes from CSV files using enforced Schema
#Schems Directive [Names as string, everything else as integers]
visitors_schema = StructType([
    StructField("visitorId", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("age", IntegerType(), True)
])

visits_schema = StructType([
    StructField("visitId", IntegerType(), True),
    StructField("visitorId", IntegerType(), True),
    StructField("visitDuration", IntegerType(), True)
])

visitors_df=spark.read.csv("visitors.csv",header=True,schema=visitors_schema)
visits_df=spark.read.csv("visits.csv",header=True,schema=visits_schema)

visitors_df.show()
visits_df.show()

+---------+---------+--------+---+
|visitorId|firstName|lastName|age|
+---------+---------+--------+---+
|      101|    Alice|   Smith| 30|
|      102|      Bob| Johnson| 45|
|      103|  Charlie|Williams| 50|
|      104|     John|   Smith| 78|
+---------+---------+--------+---+

+-------+---------+-------------+
|visitId|visitorId|visitDuration|
+-------+---------+-------------+
|      1|      101|           15|
|      2|      101|           30|
|      3|      102|           45|
|      4|      102|           30|
|      5|      104|           20|
|      6|      103|           60|
|      7|      103|           50|
|      8|      104|           45|
|      9|     NULL|           45|
|     10|      101|           30|
|     11|      103|           40|
|     12|     NULL|           30|
|     13|      104|           25|
|     15|      102|           15|
+-------+---------+-------------+



In [13]:
#Display all the valid visits
visits_df.filter(F.col("visitorId").isNotNull()).show()

#Fetch the total number of valid visits
visits_df.filter(F.col("visitorId").isNotNull()).count()


+-------+---------+-------------+
|visitId|visitorId|visitDuration|
+-------+---------+-------------+
|      1|      101|           15|
|      2|      101|           30|
|      3|      102|           45|
|      4|      102|           30|
|      5|      104|           20|
|      6|      103|           60|
|      7|      103|           50|
|      8|      104|           45|
|     10|      101|           30|
|     11|      103|           40|
|     13|      104|           25|
|     15|      102|           15|
+-------+---------+-------------+



12

In [16]:
#Get the total hours of Musuem visit
visits_df.agg(F.sum("visitDuration")).collect()[0][0]

480

In [30]:
#Fetch the top two most frequent visitors on the basis of occurances
df1=visits_df.groupby("visitorId").count().orderBy(F.desc("count")).limit(2)
df1.show()

+---------+-----+
|visitorId|count|
+---------+-----+
|      101|    3|
|      103|    3|
+---------+-----+



In [32]:
#Add a new column visitorFullName & display complete dataframe
df2=visitors_df.withColumn("visitorFullName",F.concat_ws(" ", F.col("firstName"),F.col("lastName")))
df2.show()

+---------+---------+--------+---+----------------+
|visitorId|firstName|lastName|age| visitorFullName|
+---------+---------+--------+---+----------------+
|      101|    Alice|   Smith| 30|     Alice Smith|
|      102|      Bob| Johnson| 45|     Bob Johnson|
|      103|  Charlie|Williams| 50|Charlie Williams|
|      104|     John|   Smith| 78|      John Smith|
+---------+---------+--------+---+----------------+



In [38]:
#Display the name of visitors with maximum visiting hours
df3=visits_df.groupby("visitorId").agg(F.sum("visitDuration").alias("Total")).orderBy(F.desc("Total")).limit(1)
df3.show()

+---------+-----+
|visitorId|Total|
+---------+-----+
|      103|  150|
+---------+-----+



In [47]:
#Find the visitor with maximum visiting hours in a single visit
df4=visits_df.orderBy(F.desc("visitDuration")).select(F.col("visitorId"),F.col("visitDuration")).limit(1)
df4.show()

+---------+-------------+
|visitorId|visitDuration|
+---------+-------------+
|      103|           60|
+---------+-------------+



In [53]:
#Show the visitor info according to age-seniority
visitors_df.orderBy(F.desc("age")).show()

+---------+---------+--------+---+
|visitorId|firstName|lastName|age|
+---------+---------+--------+---+
|      104|     John|   Smith| 78|
|      103|  Charlie|Williams| 50|
|      102|      Bob| Johnson| 45|
|      101|    Alice|   Smith| 30|
+---------+---------+--------+---+

